# 05 Combine all HWW and Hbb 1D fits + optional 2D plotting style match

## Notebook purpose
Combine HWW and Hbb constraints operator-by-operator.

## Combination formula
Assuming independent measurements: χ²_comb(c)=χ²_HWW(c)+χ²_Hbb(c).

Confidence intervals are extracted from Δχ² relative to the minimum.

## Output products
- Per-operator combined scan tables.
- Summary table with best-fit WC and 1σ interval bounds.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, mplhep as hep
hep.style.use('CMS')

In [ ]:
def combine_1d(df_hww,df_hbb):
    m=df_hww[['wc','chi2_hww']].merge(df_hbb[['wc','chi2_hbb']],on='wc',how='inner')
    m['chi2_comb']=m['chi2_hww']+m['chi2_hbb']
    return m.sort_values('chi2_comb')

def onesigma(df,col='chi2_comb'):
    cmin=df[col].min(); ok=df[df[col]<=cmin+1]
    return float(ok['wc'].min()),float(ok['wc'].max()),float(cmin)

In [ ]:
def combine_all_1d(HWW_ALL,HBB_ALL):
    out={}
    common=sorted(set(HWW_ALL.keys()) & set(HBB_ALL.keys()))
    for op in common: out[op]=combine_1d(HWW_ALL[op],HBB_ALL[op])
    return out

def summary_combined(COMB_ALL):
    rows=[]
    for op,df in COMB_ALL.items():
        lo,hi,cmin=onesigma(df)
        rows.append({'op':op,'best_wc':df.loc[df.chi2_comb.idxmin(),'wc'],'wc_lo_1s':lo,'wc_hi_1s':hi,'chi2_min':cmin})
    return pd.DataFrame(rows).sort_values('chi2_min')